In [0]:
# ========================================
# EXPLORATORY NOTEBOOK
# DATASET: gold_dim_calendar
# ========================================

# ========================================
# 0. CONFIGURATION
# ========================================

CATALOG = "ptfrozenfoods_dev"
SOURCE_SCHEMA = "silver"
TARGET_SCHEMA = "gold"

DOMAIN = "dimensions"
DATASET = "gold_dim_calendar"

STORAGE_ACCOUNT = "stptfrozenfoodsdevwe01"
SILVER_CONTAINER = "silver"
GOLD_CONTAINER = "gold"

CALENDAR_DATASET = "reference_calendar"

CALENDAR_TABLE = f"{CATALOG}.{SOURCE_SCHEMA}.{CALENDAR_DATASET}"

CANDIDATE_TARGET_TABLE = f"{CATALOG}.{TARGET_SCHEMA}.dim_calendar"

SILVER_BASE_PATH = f"abfss://{SILVER_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
GOLD_BASE_PATH = f"abfss://{GOLD_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"

CALENDAR_SOURCE_PATH = f"{SILVER_BASE_PATH}reference/{CALENDAR_DATASET}/"
CANDIDATE_TARGET_PATH = f"{GOLD_BASE_PATH}dimensions/dim_calendar/"

In [0]:
# ========================================
# 1. CONTEXT SETUP
# ========================================

print("=" * 80)
print("STARTING GOLD EXPLORATORY: gold_dim_calendar")
print("=" * 80)

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SOURCE_SCHEMA}")

print("[INFO] Context setup completed successfully.")

STARTING GOLD EXPLORATORY: gold_dim_calendar
[INFO] Context setup completed successfully.


In [0]:
# ========================================
# 2. CONFIGURATION SUMMARY
# ========================================

print("=" * 80)
print("GOLD EXPLORATORY NOTEBOOK CONFIGURATION")
print("=" * 80)
print(f"Catalog:                         {CATALOG}")
print(f"Source schema:                   {SOURCE_SCHEMA}")
print(f"Target schema:                   {TARGET_SCHEMA}")
print(f"Domain:                          {DOMAIN}")
print(f"Dataset:                         {DATASET}")
print(f"Calendar table:                  {CALENDAR_TABLE}")
print(f"Candidate target table:          {CANDIDATE_TARGET_TABLE}")
print(f"Calendar source path:            {CALENDAR_SOURCE_PATH}")
print(f"Candidate target path:           {CANDIDATE_TARGET_PATH}")
print("=" * 80)

GOLD EXPLORATORY NOTEBOOK CONFIGURATION
Catalog:                         ptfrozenfoods_dev
Source schema:                   silver
Target schema:                   gold
Domain:                          dimensions
Dataset:                         gold_dim_calendar
Calendar table:                  ptfrozenfoods_dev.silver.reference_calendar
Candidate target table:          ptfrozenfoods_dev.gold.dim_calendar
Calendar source path:            abfss://silver@stptfrozenfoodsdevwe01.dfs.core.windows.net/reference/reference_calendar/
Candidate target path:           abfss://gold@stptfrozenfoodsdevwe01.dfs.core.windows.net/dimensions/dim_calendar/


In [0]:
# ========================================
# 3. PRE-CHECKS
# ========================================

print("[INFO] Checking source table availability...")
spark.sql(f"DESCRIBE TABLE {CALENDAR_TABLE}")

print("[INFO] Checking source path access...")
dbutils.fs.ls(CALENDAR_SOURCE_PATH)

print("[INFO] Checking target container access...")
dbutils.fs.ls(f"abfss://{GOLD_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/")

print("[INFO] Pre-checks completed successfully.")

[INFO] Checking source table availability...
[INFO] Checking source path access...
[INFO] Checking target container access...
[INFO] Pre-checks completed successfully.


In [0]:
# ========================================
# 4. READ SOURCE DATA
# ========================================

print("[INFO] Loading calendar data from the Silver layer...")

df_calendar = spark.table(CALENDAR_TABLE)

print("[INFO] Source tables loaded successfully.")

print("=" * 80)
print("SOURCE DATA ROW COUNTS")
print("=" * 80)
print(f"Calendar: {df_calendar.count():,}")
print("=" * 80)

# ========================================
# DATASET PREVIEW — INITIAL EXPLORATION
# ========================================

print("=" * 80)
print("DATASET PREVIEW — INITIAL EXPLORATION")
print("=" * 80)

print("\n[INFO] Preview: CALENDAR (df_calendar)")
print("-" * 80)
display(df_calendar.limit(5))

print("\n[INFO] Printing schema...")
df_calendar.printSchema()

print("\n" + "=" * 80)
print("[INFO] Dataset preview completed successfully.")
print("=" * 80)

[INFO] Loading calendar data from the Silver layer...
[INFO] Source tables loaded successfully.
SOURCE DATA ROW COUNTS
Calendar: 1,885
DATASET PREVIEW — INITIAL EXPLORATION

[INFO] Preview: CALENDAR (df_calendar)
--------------------------------------------------------------------------------


data,ano,mes,dia,trimestre,nome_mes,dia_semana,nome_dia_semana,is_fim_de_semana,is_inicio_mes,is_fim_mes,load_date,ingestion_timestamp,source_file
2021-01-13,2021,1,13,1,Janeiro,3,Quarta-feira,0,0,0,2026-03-19,2026-04-01T22:04:54.555Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/reference/reference_calendar/load_date=2026-03-19/reference_calendar_20260319T151315Z.csv
2021-01-16,2021,1,16,1,Janeiro,6,Sábado,1,0,0,2026-03-19,2026-04-01T22:04:54.555Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/reference/reference_calendar/load_date=2026-03-19/reference_calendar_20260319T151315Z.csv
2021-01-27,2021,1,27,1,Janeiro,3,Quarta-feira,0,0,1,2026-03-19,2026-04-01T22:04:54.555Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/reference/reference_calendar/load_date=2026-03-19/reference_calendar_20260319T151315Z.csv
2021-02-04,2021,2,4,1,Fevereiro,4,Quinta-feira,0,1,0,2026-03-19,2026-04-01T22:04:54.555Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/reference/reference_calendar/load_date=2026-03-19/reference_calendar_20260319T151315Z.csv
2021-04-05,2021,4,5,2,Abril,1,Segunda-feira,0,1,0,2026-03-19,2026-04-01T22:04:54.555Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/reference/reference_calendar/load_date=2026-03-19/reference_calendar_20260319T151315Z.csv



[INFO] Printing schema...
root
 |-- data: date (nullable = true)
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- dia: integer (nullable = true)
 |-- trimestre: integer (nullable = true)
 |-- nome_mes: string (nullable = true)
 |-- dia_semana: integer (nullable = true)
 |-- nome_dia_semana: string (nullable = true)
 |-- is_fim_de_semana: integer (nullable = true)
 |-- is_inicio_mes: integer (nullable = true)
 |-- is_fim_mes: integer (nullable = true)
 |-- load_date: date (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)


[INFO] Dataset preview completed successfully.


In [0]:
# ========================================
# 5. SOURCE VALIDATION
# ========================================

from pyspark.sql import functions as F

required_columns = [
    "data",
    "ano",
    "mes",
    "dia"
]

missing_columns = [c for c in required_columns if c not in df_calendar.columns]

print(f"[INFO] Missing required columns: {missing_columns}")

if missing_columns:
    raise Exception(f"Missing required columns: {missing_columns}")

print("[INFO] Source validation completed successfully.")

[INFO] Missing required columns: []
[INFO] Source validation completed successfully.


In [0]:
# ========================================
# 6. INITIAL DATA QUALITY CHECKS
# ========================================

display(
    df_calendar.select(
        F.count("*").alias("total_rows"),
        F.countDistinct("data").alias("distinct_dates"),
        F.sum(F.when(F.col("data").isNull(), 1).otherwise(0)).alias("null_dates")
    )
)

total_rows,distinct_dates,null_dates
1885,1885,0


In [0]:
# ========================================
# 7. DUPLICATE CHECK — CALENDAR
# ========================================

df_duplicates = (
    df_calendar
    .groupBy("data")
    .count()
    .filter(F.col("count") > 1)
)

display(df_duplicates)

data,count


In [0]:
# ========================================
# 8. BUSINESS PROFILE CHECKS
# ========================================

print("[INFO] Year distribution")
display(df_calendar.groupBy("ano").count().orderBy("ano"))

print("[INFO] Month distribution")
display(df_calendar.groupBy("nome_mes").count().orderBy("count", ascending=False))

print("[INFO] Quarter distribution")
display(df_calendar.groupBy("trimestre").count().orderBy("trimestre"))

print("[INFO] Weekend distribution")
display(df_calendar.groupBy("is_fim_de_semana").count())

[INFO] Year distribution


ano,count
2021,365
2022,365
2023,365
2024,366
2025,365
2026,59


[INFO] Month distribution


nome_mes,count
Janeiro,186
Fevereiro,169
Dezembro,155
Julho,155
Agosto,155
Maio,155
Outubro,155
Março,155
Novembro,150
Setembro,150


[INFO] Quarter distribution


trimestre,count
1,510
2,455
3,460
4,460


[INFO] Weekend distribution


is_fim_de_semana,count
1,539
0,1346


In [0]:
# ========================================
# 9. BUILD CANDIDATE DIMENSION
# ========================================

df_dim_calendar_candidate = (
    df_calendar
    .select(
        F.col("data").alias("calendar_date"),
        F.col("ano").alias("calendar_year"),
        F.col("mes").alias("calendar_month"),
        F.col("dia").alias("calendar_day"),
        F.col("trimestre").alias("calendar_quarter"),
        F.col("nome_mes").alias("calendar_month_name"),
        F.col("dia_semana").alias("calendar_day_of_week"),
        F.col("nome_dia_semana").alias("calendar_day_name"),
        F.col("is_fim_de_semana").alias("calendar_is_weekend"),
        F.col("is_inicio_mes").alias("calendar_is_month_start"),
        F.col("is_fim_mes").alias("calendar_is_month_end")
    )
    .dropDuplicates(["calendar_date"])
)

print(f"[INFO] Candidate dimension row count: {df_dim_calendar_candidate.count():,}")
display(df_dim_calendar_candidate)

[INFO] Candidate dimension row count: 1,885


calendar_date,calendar_year,calendar_month,calendar_day,calendar_quarter,calendar_month_name,calendar_day_of_week,calendar_day_name,calendar_is_weekend,calendar_is_month_start,calendar_is_month_end
2021-01-01,2021,1,1,1,Janeiro,5,Sexta-feira,0,1,0
2021-01-02,2021,1,2,1,Janeiro,6,Sábado,1,1,0
2021-01-03,2021,1,3,1,Janeiro,7,Domingo,1,1,0
2021-01-04,2021,1,4,1,Janeiro,1,Segunda-feira,0,1,0
2021-01-05,2021,1,5,1,Janeiro,2,Terça-feira,0,1,0
2021-01-06,2021,1,6,1,Janeiro,3,Quarta-feira,0,0,0
2021-01-07,2021,1,7,1,Janeiro,4,Quinta-feira,0,0,0
2021-01-08,2021,1,8,1,Janeiro,5,Sexta-feira,0,0,0
2021-01-09,2021,1,9,1,Janeiro,6,Sábado,1,0,0
2021-01-10,2021,1,10,1,Janeiro,7,Domingo,1,0,0


In [0]:
# ========================================
# 10. CANDIDATE DIMENSION VALIDATION
# ========================================

display(
    df_dim_calendar_candidate.select(
        F.count("*").alias("total_rows"),
        F.countDistinct("calendar_date").alias("distinct_dates"),
        F.sum(F.when(F.col("calendar_date").isNull(), 1).otherwise(0)).alias("null_calendar_date")
    )
)

print("[INFO] Candidate dimension validation completed successfully.")

total_rows,distinct_dates,null_calendar_date
1885,1885,0


[INFO] Candidate dimension validation completed successfully.


In [0]:
# ========================================
# 11. MISSING VALUES ANALYSIS — DIM CALENDAR
# ========================================

from pyspark.sql import functions as F

def analyze_missing_values(df, dataset_name):
    print("=" * 80)
    print(f"MISSING VALUES ANALYSIS — {dataset_name.upper()}")
    print("=" * 80)

    total_rows = df.count()

    missing_values_df = df.select([
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df.columns
    ])

    missing_values_transposed = (
        missing_values_df
        .select(
            F.array([
                F.struct(
                    F.lit(col_name).alias("column_name"),
                    F.col(col_name).alias("null_count")
                ) for col_name in missing_values_df.columns
            ]).alias("missing_data")
        )
        .select(F.explode("missing_data").alias("data"))
        .select(
            F.col("data.column_name"),
            F.col("data.null_count")
        )
        .withColumn(
            "null_percentage",
            F.round((F.col("null_count") / F.lit(total_rows)) * 100, 4)
        )
        .orderBy(F.col("null_percentage").desc())
    )

    display(missing_values_transposed)

    print(f"[INFO] Total rows analyzed: {total_rows:,}")
    print("[INFO] Missing values analysis completed successfully.")
    print("=" * 80)


analyze_missing_values(df_dim_calendar_candidate, "dim_calendar")

MISSING VALUES ANALYSIS — DIM_CALENDAR


column_name,null_count,null_percentage
calendar_day_name,0,0.0
calendar_quarter,0,0.0
calendar_date,0,0.0
calendar_month,0,0.0
calendar_month_name,0,0.0
calendar_year,0,0.0
calendar_is_month_start,0,0.0
calendar_is_weekend,0,0.0
calendar_day_of_week,0,0.0
calendar_day,0,0.0


[INFO] Total rows analyzed: 1,885
[INFO] Missing values analysis completed successfully.
